# Lab 3 — Personal Assistant with Qwen3.5
<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 250px; height: 150px; vertical-align: middle;">
            <img src="../assets/logo.png" width="250" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Objective</h2>
            <span style="color:#ff7800;">
- Compare performance with Llama3.2<br>
- Integration and Execution of tools with Qwen<br>
- Application execution and Gradio deployment
            </span>
        </td>
    </tr>
</table>

## How to Use This Notebook
This lab builds the same personal assistant as Lab 3, but uses the Qwen3.5 model instead of Llama3.2. Run the cells in order and verify each capability before moving on.

1. Load environment variables and test Pushover notifications.
2. Define the tools and confirm the tool handler returns results.
3. Load personal context from files and build the system prompt.
4. Start the Gradio app with Qwen3.5 and test real conversations end to end.

## Qwen3.5 vs Llama3.2
Qwen3.5 is Alibaba's multilingual Large Language Model. You may notice differences in:
- Response style and tone
- Tool calling behavior
- Response speed
- Language handling

This is a great opportunity to compare different models and their trade-offs!

In [23]:

from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from ollama import Client
from pypdf import PdfReader
import gradio as gr
client = Client(host='http://localhost:11434')

In [24]:
load_dotenv()
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = 'https://api.pushover.net/1/messages.json'

if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with a


In [25]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    response = requests.post(pushover_url, data=payload)

In [26]:
push("Hello from Agentic AI Lab 4 with Qwen3.5!")

Push: Hello from Agentic AI Lab 4 with Qwen3.5!


In [27]:
def record_user_details(email, name="Name not provided", notes="not provided"):
    push(f"Recording interest from {name} with email {email} and notes {notes}")
    return {"recorded": "ok"}

def record_unknown_question(question):
    push(f"Recording {question} asked that I couldn't answer")
    return {"recorded": "ok"}

In [ ]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The email address of this user"
            },
            "name": {
                "type": "string",
                "description": "The user's name, if they provided it"
            }
            ,
            "notes": {
                "type": "string",
                "description": "Any additional information about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [ ]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that couldn't be answered"
            },
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [28]:
tools = [{"type": "function", "function": record_user_details_json},
        {"type": "function", "function": record_unknown_question_json}]

In [29]:
tools

[{'type': 'function',
  'function': {'name': 'record_user_details',
   'description': 'Use this tool to record that a user is interested in being in touch and provided an email address',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'The email address of this user'},
     'name': {'type': 'string',
      'description': "The user's name, if they provided it"},
     'notes': {'type': 'string',
      'description': "Any additional information about the conversation that's worth recording to give context"}},
    'required': ['email'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'record_unknown_question',
   'description': "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
   'parameters': {'type': 'object',
    'properties': {'question': {'type': 'string',
      'description': "The question that couldn't be answered"}},
    'required': ['quest

In [30]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.get('function', {}).get('name', '')
        arguments = tool_call.get('function', {}).get('arguments', {})
        
        if isinstance(arguments, str):
            arguments = json.loads(arguments)
        
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        
        results.append({
            "role": "tool",
            "content": json.dumps(result)
        })
    return results

In [31]:
reader = PdfReader(".././me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open(".././me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

name = "Abhishek"

In [32]:
system_prompt = f"""You are a helpful assistant that provides information about {name} based on the information provided in the LinkedIn Profile and Summary.\
        ## Summary \n {summary} \n \n ## LinkedIn Profile \n {linkedin}.\
        You have to assume character of {name} and reply as if you are {name}.\
        With this context, answer the question as best as you can.\
        Never say you are a language model or AI. Speak in first person as {name}.\
        
        TOOL USAGE RULES:
        1. For personal preference questions you don't know (like food needs, dietary preferences):
           - Respond politely: "That's a great question! I'd love to discuss that with you. Could you share your email so I can get back to you on that?"
           - If they provide an email address, respond with: "Thanks for your email! I'll be in touch about that." and ALWAYS call both record_user_details tool (with their email) and record_unknown_question tool.
           - If they don't provide email, ask again: "Could you share your email address so we can continue this conversation?" Do NOT call tools yet.
           - When they eventually provide email, extract it and call record_user_details with notes="Inquiry about food preferences"
        
        2. If you do not know the answer to a general question (and they didn't explicitly ask about personal preferences), respond with EXACTLY: "I don't know" and ALWAYS call the record_unknown_question tool.
        
        3. If the user provides an email address and explicitly asks to be contacted for other reasons, respond with EXACTLY: "Email is recorded and I will contact you soon." and ALWAYS call the record_user_details tool.
        
        4. If the user asks to be contacted but did not provide an email, ask them to share their email address. Do NOT call any tool.
        
        5. For all other cases (greetings, questions you can answer, general conversation), reply normally and do NOT call any tool.
        
        Important: Only call tools when your response matches one of the exact phrases above. Never call tools for regular conversation or when you have provided an answer to a question.
        Chat with the user always staying in the character of an assistant providing information about {name}.
        """


In [ ]:
def chat_with_me(message,chat_history):
        # Extract text content from Gradio message structure
    if isinstance(message, dict) and 'text' in message:
        user_message = message['text']
    elif isinstance(message, list) and len(message) > 0 and isinstance(message[0], dict):
        user_message = message[0].get('text', str(message))
    else:
        user_message = str(message)
    
    # Build messages array with system prompt
    messages = [{"role": "system", "content": system_prompt}]

    for msg in chat_history:
        if isinstance(msg, dict):
            # Handle dict format with role and content
            if 'role' in msg and 'content' in msg:
                content = msg['content']
                if isinstance(content, list):
                    content = content[0].get('text', '') if content else ''
                elif isinstance(content, dict):
                    content = content.get('text', str(content))
                messages.append({"role": msg['role'], "content": str(content)})
        elif isinstance(msg, (list, tuple)) and len(msg) >= 2:
            # Handle tuple format (user_msg, assistant_msg)
            user_msg = msg[0]
            assistant_msg = msg[1]
            
            # Extract text from user message
            if isinstance(user_msg, dict):
                user_msg = user_msg.get('text', str(user_msg))
            messages.append({"role": "user", "content": str(user_msg)})
            
            # Extract text from assistant message
            if assistant_msg:
                if isinstance(assistant_msg, dict):
                    assistant_msg = assistant_msg.get('text', str(assistant_msg))
                messages.append({"role": "assistant", "content": str(assistant_msg)})

    #Append Current user message
    messages.append({"role": "user", "content": user_message})

    done = False
    while not done:
        response = client.chat(
            model='qwen3:8b',
            messages=messages,
            tools=tools
        )
        
        # Check if there are tool calls
        tool_calls = response['message'].get('tool_calls')
        
        if tool_calls:
            # Add assistant message with tool calls to history
            messages.append(response['message'])
            # Handle tool calls and add results
            tool_results = handle_tool_calls(tool_calls)
            messages.extend(tool_results)
        else:
            done = True
    
    return response['message']['content']

In [ ]:
gr.ChatInterface(chat_with_me).launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


Tool called: record_user_details
Push: Recording interest from Name not provided with email dummy@example.com and notes Inquiry about food preferences
